## Stream Customers Data From Cloud Files to Delta Lake

1. Read files from cloud storage using DataStreamReader API
2. Transform the dataframe to add the following columns
    - file path: Cloud file path
    - ingestion date: Current timestamp
3. Write the transformed data stream to Delta Lake Table

## 1. Read files using Auto loader

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType

cusstomers_schema = StructType(
    fields = [
        StructField('customer_id', IntegerType()),
        StructField('customer_name', StringType()),
        StructField('date_of_birth', DateType()),
        StructField('telephone', StringType()),
        StructField('email', StringType()),
        StructField('member_since', StringType()),
        StructField('created_timestamp', TimestampType())
    ]
)


In [0]:
customers_df = (spark.readStream 
     .format('json') 
     .schema(cusstomers_schema) 
     .load("/Volumes/gizmobox/landing/operational_data/customers_stream/") 
     )


## 2. Transform the dataframe to add the following columns
        - file path: Cloud file path
        - ingestion date: Current timestamp

In [0]:
from pyspark.sql import functions as F

cunstomers_transformed_df = (customers_df.withColumn("file_path",F.col("_metadata.file_path"))
            .withColumn("ingestion_date", F.current_timestamp())
)

## 3. Write the transformed data stream to Delta Lake Table

In [0]:
streaming_query = (
    cunstomers_transformed_df.writeStream 
        .format("delta") 
        .option("checkpointLocation", "abfss://gizmobox@deacourseexternaldl.dfs.core.windows.net/landing/operational_data/customers_stream/_checkpoint_stream")
        .toTable("gizmobox.bronze.customers_stream")
)

In [0]:
%fs ls 'abfss://gizmobox@deacourseexternaldl.dfs.core.windows.net/landing/operational_data/'

In [0]:
%sql
select * from gizmobox.bronze.customers_stream

In [0]:
streaming_query.stop()